# Paper Summary Tables

Load all available LLM-judge analyses and prepare ensemble tables for the paper. Missing or unfinished Qwen artifacts are skipped independently, so completed metric summaries can be used even before Qwen pairwise win-rate evaluation finishes.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

REPO_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "evaluation_results.py").exists())
BASE_DIR = REPO_ROOT / "evaluations" / "chatbs-base"
ANSWER_REPORT = "original"
WINRATE_DIRNAME = f"answer_winrate-{ANSWER_REPORT}"
JUDGE_IDS = ["old-llama-70b", "gpt-5.4-mini", "qwen3.6-35b-a3b"]
JUDGE_LABELS = {
    "old-llama-70b": "Llama 3.3 70B",
    "gpt-5.4-mini": "GPT-5.4 mini",
    "qwen3.6-35b-a3b": "Qwen 3.6 35B",
}

TABLE_OUTPUT_DIR = BASE_DIR / "paper_tables" / "judges" / "ensemble"
TABLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def read_csv_or_empty(path):
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()


def judge_analysis_dir(judge_id):
    return BASE_DIR / "analysis" / "judges" / judge_id

BASE_DIR

PosixPath('/home/desild/work/research/LLM-Workflow-Explorer/evaluations/chatbs-base')

## Load and Aggregate Judge Results

Each `*_mean` value is the equal-weight mean of the available judge-level means. Each corresponding `*_std` is the sample standard deviation across those judge-level means, not the within-question standard deviation stored in an individual judge's source summary. Per-judge source rows are retained in `judge_sources` and exported below.

In [2]:
CATEGORY_NAMES = ["bool", "entity", "numeric"]
availability_rows = []
per_judge_summaries = {category: [] for category in CATEGORY_NAMES}
winrate_frames = []
pairwise_frames = []

for judge_id in JUDGE_IDS:
    analysis_dir = judge_analysis_dir(judge_id)
    for category in CATEGORY_NAMES:
        path = analysis_dir / category / "results_summary.csv"
        frame = read_csv_or_empty(path)
        availability_rows.append({
            "judge_id": judge_id,
            "judge": JUDGE_LABELS[judge_id],
            "artifact": f"{category}_summary",
            "available": not frame.empty,
            "path": str(path.relative_to(REPO_ROOT)),
        })
        if not frame.empty:
            frame.insert(0, "judge_id", judge_id)
            frame.insert(1, "judge", JUDGE_LABELS[judge_id])
            per_judge_summaries[category].append(frame)

    winrate_path = analysis_dir / WINRATE_DIRNAME / "answer_winrate_summary.csv"
    winrate = read_csv_or_empty(winrate_path)
    availability_rows.append({
        "judge_id": judge_id,
        "judge": JUDGE_LABELS[judge_id],
        "artifact": "answer_winrate_summary",
        "available": not winrate.empty,
        "path": str(winrate_path.relative_to(REPO_ROOT)),
    })
    if not winrate.empty:
        winrate.insert(0, "judge_id", judge_id)
        winrate.insert(1, "judge", JUDGE_LABELS[judge_id])
        winrate_frames.append(winrate)

    pairwise_path = analysis_dir / WINRATE_DIRNAME / "pairwise_answer_winrate.csv"
    pairwise = read_csv_or_empty(pairwise_path)
    availability_rows.append({
        "judge_id": judge_id,
        "judge": JUDGE_LABELS[judge_id],
        "artifact": "pairwise_answer_winrate",
        "available": not pairwise.empty,
        "path": str(pairwise_path.relative_to(REPO_ROOT)),
    })
    if not pairwise.empty:
        pairwise["judge_id"] = judge_id
        pairwise["judge"] = JUDGE_LABELS[judge_id]
        pairwise_frames.append(pairwise)

availability = pd.DataFrame(availability_rows)


def aggregate_judge_values(source, group_column, metric_columns):
    rows = []
    for group_name, group in source.groupby(group_column, dropna=False):
        counts = pd.to_numeric(group["evaluated_examples"], errors="coerce")
        row = {
            group_column: group_name,
            "judge_count": group["judge_id"].nunique(),
            "evaluated_examples": counts.min(),
            "evaluated_examples_min": counts.min(),
            "evaluated_examples_max": counts.max(),
        }
        for column in metric_columns:
            values = pd.to_numeric(group[column], errors="coerce")
            row[column] = values.mean()
            row[column.removesuffix("_mean") + "_std"] = values.std(ddof=1)
        rows.append(row)
    return pd.DataFrame(rows)


judge_sources = {}
summaries = {}
for category in CATEGORY_NAMES:
    if not per_judge_summaries[category]:
        continue
    source = pd.concat(per_judge_summaries[category], ignore_index=True)
    metric_columns = [column for column in source.columns if column.endswith("_mean")]
    judge_sources[category] = source
    summaries[category] = aggregate_judge_values(source, "run", metric_columns)

if winrate_frames:
    winrate_source = pd.concat(winrate_frames, ignore_index=True)
    winrate_rows = []
    for method, group in winrate_source.groupby("method", dropna=False):
        row = {
            "method": method,
            "judge_count": group["judge_id"].nunique(),
            "comparisons_min": group["comparisons"].min(),
            "comparisons_max": group["comparisons"].max(),
        }
        for column in ["wins", "losses", "ties", "winrate"]:
            values = pd.to_numeric(group[column], errors="coerce")
            row[f"{column}_mean"] = values.mean()
            row[f"{column}_std"] = values.std(ddof=1)
        winrate_rows.append(row)
    summaries["answer_winrate"] = pd.DataFrame(winrate_rows)
else:
    winrate_source = pd.DataFrame()

if pairwise_frames:
    summaries["pairwise_answer_winrate"] = pd.concat(pairwise_frames, ignore_index=True)

display(availability)
summaries.keys()

,judge_id,judge,artifact,available,path
0,old-llama-70b,Llama 3.3 70B,bool_summary,True,evaluations/chatbs-base/analysis/judges/old-ll...
1,old-llama-70b,Llama 3.3 70B,entity_summary,True,evaluations/chatbs-base/analysis/judges/old-ll...
2,old-llama-70b,Llama 3.3 70B,numeric_summary,True,evaluations/chatbs-base/analysis/judges/old-ll...
3,old-llama-70b,Llama 3.3 70B,answer_winrate_summary,True,evaluations/chatbs-base/analysis/judges/old-ll...
4,old-llama-70b,Llama 3.3 70B,pairwise_answer_winrate,True,evaluations/chatbs-base/analysis/judges/old-ll...
5,gpt-5.4-mini,GPT-5.4 mini,bool_summary,True,evaluations/chatbs-base/analysis/judges/gpt-5....
6,gpt-5.4-mini,GPT-5.4 mini,entity_summary,True,evaluations/chatbs-base/analysis/judges/gpt-5....
7,gpt-5.4-mini,GPT-5.4 mini,numeric_summary,True,evaluations/chatbs-base/analysis/judges/gpt-5....
8,gpt-5.4-mini,GPT-5.4 mini,answer_winrate_summary,True,evaluations/chatbs-base/analysis/judges/gpt-5....
9,gpt-5.4-mini,GPT-5.4 mini,pairwise_answer_winrate,True,evaluations/chatbs-base/analysis/judges/gpt-5....


dict_keys(['bool', 'entity', 'numeric', 'answer_winrate', 'pairwise_answer_winrate'])

In [3]:
# Quick preview of each loaded summary.
for name, df in summaries.items():
    print(f"\n{name}: {df.shape[0]} rows x {df.shape[1]} columns")
    display(df.head())


bool: 7 rows x 35 columns


,run,judge_count,evaluated_examples,evaluated_examples_min,evaluated_examples_max,answer_token_precision_mean,answer_token_precision_std,answer_token_recall_mean,answer_token_recall_std,answer_token_f1_mean,answer_token_f1_std,gt_entity_total_mean,gt_entity_total_std,gt_entity_covered_mean,gt_entity_covered_std,gt_entity_coverage_mean,gt_entity_coverage_std,bertscore_precision_mean,bertscore_precision_std,bertscore_recall_mean,bertscore_recall_std,bertscore_f1_mean,bertscore_f1_std,llm_completeness_mean,llm_completeness_std,llm_faithfulness_mean,llm_faithfulness_std,llm_relevance_mean,llm_relevance_std,llm_understanderbility_mean,llm_understanderbility_std,nli_entailment_max_mean,nli_entailment_max_std,bool_accuracy_mean,bool_accuracy_std
0,fullcontext,3,38,38,38,0.637632,0.0,0.472221,0.0,0.496864,0.000000e+00,2.289474,0.0,0.394737,0.0,0.146592,0.000000e+00,0.808490,0.000000e+00,0.836027,0.000000e+00,0.821554,0.000000e+00,0.821053,0.022942,0.824561,0.018900,0.973509,0.016646,0.913684,0.101366,0.655778,0.0,0.894737,0.0
1,grasp,3,38,38,38,0.106623,0.0,0.093875,0.0,0.085124,1.699675e-17,2.289474,0.0,0.026316,0.0,0.002024,0.000000e+00,0.840972,0.000000e+00,0.766515,1.359740e-16,0.800113,1.359740e-16,0.395614,0.224601,0.409649,0.208033,0.614912,0.241480,0.758596,0.209620,0.492698,0.0,0.289474,0.0
2,hipporag,3,38,38,38,0.000000,0.0,0.000000,0.0,0.000000,0.000000e+00,2.289474,0.0,0.289474,0.0,0.206140,3.399350e-17,0.709924,0.000000e+00,0.762432,0.000000e+00,0.735114,0.000000e+00,0.346266,0.047563,0.413383,0.008927,0.936266,0.025926,0.978709,0.031817,0.260601,0.0,0.526316,0.0
3,hypergraphrag,3,38,38,38,0.236746,0.0,0.418532,0.0,0.258888,0.000000e+00,2.289474,0.0,0.052632,0.0,0.008603,0.000000e+00,0.833537,1.359740e-16,0.806944,0.000000e+00,0.819021,0.000000e+00,0.376754,0.014553,0.494474,0.186590,0.543158,0.005076,0.917193,0.066243,0.503595,0.0,0.421053,0.0
4,llmbased,3,38,38,38,0.347558,0.0,0.165251,0.0,0.205004,0.000000e+00,2.289474,0.0,0.000000,0.0,0.000000,0.000000e+00,0.781740,0.000000e+00,0.776166,1.359740e-16,0.778511,0.000000e+00,0.127489,0.039468,0.105512,0.030269,0.233286,0.124887,0.696515,0.085003,0.277763,0.0,0.000000,0.0



entity: 7 rows x 51 columns


,run,judge_count,evaluated_examples,evaluated_examples_min,evaluated_examples_max,answer_token_precision_mean,answer_token_precision_std,answer_token_recall_mean,answer_token_recall_std,answer_token_f1_mean,answer_token_f1_std,gt_entity_total_mean,gt_entity_total_std,gt_entity_covered_mean,gt_entity_covered_std,gt_entity_coverage_mean,gt_entity_coverage_std,bertscore_precision_mean,bertscore_precision_std,bertscore_recall_mean,bertscore_recall_std,bertscore_f1_mean,bertscore_f1_std,llm_completeness_mean,llm_completeness_std,llm_faithfulness_mean,llm_faithfulness_std,llm_relevance_mean,llm_relevance_std,llm_understanderbility_mean,llm_understanderbility_std,nli_entailment_max_mean,nli_entailment_max_std,entity_gt_total_mean,entity_gt_total_std,entity_retrieved_final_total_mean,entity_retrieved_final_total_std,entity_retrieved_total_total_mean,entity_retrieved_total_total_std,entity_recall_final_mean,entity_recall_final_std,entity_precision_final_mean,entity_precision_final_std,entity_f1_final_mean,entity_f1_final_std,entity_recall_total_mean,entity_recall_total_std,entity_precision_total_mean,entity_precision_total_std,entity_f1_total_mean,entity_f1_total_std
0,fullcontext,3,49,49,49,0.410651,0.0,0.285590,0.0,0.250826,0.0,2.673469,0.0,0.428571,6.798700e-17,0.350775,0.000000,0.834169,0.0,0.843230,1.359740e-16,0.838148,1.359740e-16,0.613472,0.092350,0.651831,0.068256,0.824209,0.042767,0.850510,0.098733,0.314016,0.0,2.55102,0.0,6.857143,0.000000,11.142857,2.175584e-15,0.566949,0.000000e+00,0.160490,0.0,0.379824,0.0,0.658386,0.000000,0.112145,1.699675e-17,0.268076,0.0
1,grasp,3,49,49,49,0.185071,0.0,0.112896,0.0,0.097409,0.0,2.673469,0.0,0.054422,5.891329e-02,0.016150,0.024616,0.826952,0.0,0.786332,0.000000e+00,0.805476,0.000000e+00,0.191837,0.102056,0.240751,0.087932,0.311341,0.141013,0.607183,0.257871,0.260185,0.0,2.55102,0.0,0.156463,0.271001,0.156463,2.710011e-01,0.009044,1.566454e-02,0.175000,NaN,0.583333,NaN,0.009044,0.015665,0.175000,NaN,0.583333,NaN
2,hipporag,3,49,49,49,0.270040,0.0,0.114260,0.0,0.116263,0.0,2.673469,0.0,0.265306,0.000000e+00,0.053488,0.000000,0.719528,0.0,0.781964,1.359740e-16,0.749001,0.000000e+00,0.194039,0.061470,0.196421,0.057843,0.401978,0.156927,0.752541,0.061698,0.201577,0.0,2.55102,0.0,79.979592,0.000000,158.102041,0.000000e+00,0.345458,6.798700e-17,0.009490,0.0,0.044388,0.0,0.647933,0.000000,0.013366,2.124594e-18,0.041964,0.0
3,hypergraphrag,3,49,49,49,0.203279,0.0,0.300364,0.0,0.162739,0.0,2.673469,0.0,0.183673,0.000000e+00,0.060078,0.000000,0.810532,0.0,0.801310,0.000000e+00,0.805147,0.000000e+00,0.383485,0.085850,0.446438,0.068321,0.618410,0.073523,0.824289,0.066603,0.286587,0.0,2.55102,0.0,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000e+00,NaN,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN
4,llmbased,3,49,49,49,0.318264,0.0,0.149108,0.0,0.143759,0.0,2.673469,0.0,0.000000,0.000000e+00,0.000000,0.000000,0.783663,0.0,0.779654,0.000000e+00,0.781073,0.000000e+00,0.175850,0.024025,0.179592,0.058333,0.345918,0.040316,0.666871,0.103368,0.145957,0.0,2.55102,0.0,9.224490,0.000000,11.489796,0.000000e+00,0.298974,0.000000e+00,0.045294,0.0,0.163553,0.0,0.568353,0.000000,0.085533,0.000000e+00,0.196742,0.0



numeric: 7 rows x 39 columns


,run,judge_count,evaluated_examples,evaluated_examples_min,evaluated_examples_max,answer_token_precision_mean,answer_token_precision_std,answer_token_recall_mean,answer_token_recall_std,answer_token_f1_mean,answer_token_f1_std,gt_entity_total_mean,gt_entity_total_std,gt_entity_covered_mean,gt_entity_covered_std,gt_entity_coverage_mean,gt_entity_coverage_std,bertscore_precision_mean,bertscore_precision_std,bertscore_recall_mean,bertscore_recall_std,bertscore_f1_mean,bertscore_f1_std,llm_completeness_mean,llm_completeness_std,llm_faithfulness_mean,llm_faithfulness_std,llm_relevance_mean,llm_relevance_std,llm_understanderbility_mean,llm_understanderbility_std,nli_entailment_max_mean,nli_entailment_max_std,numeric_ground_truth_count_mean,numeric_ground_truth_count_std,numeric_predicted_count_mean,numeric_predicted_count_std,numeric_accuracy_mean,numeric_accuracy_std
0,fullcontext,3,16,16,16,0.660500,0.0,0.345912,6.798700e-17,0.408937,6.798700e-17,2.8125,0.0,0.8125,0.0,0.125,0.0,0.824401,0.0,0.831342,0.0,0.827059,0.0,0.732639,0.039105,0.725000,0.038017,0.980208,0.014091,0.898750,0.128896,0.286214,0.000000e+00,2.8125,0.0,679.666667,0.0,0.0625,0.0
1,grasp,3,16,16,16,0.190341,0.0,0.049062,0.000000e+00,0.055236,0.000000e+00,2.8125,0.0,0.0000,0.0,0.000,0.0,0.820224,0.0,0.793875,0.0,0.804871,0.0,0.354167,0.344223,0.395833,0.308305,0.558333,0.365309,0.685000,0.294830,0.465113,0.000000e+00,2.8125,0.0,0.600000,0.0,0.1250,0.0
2,hipporag,3,16,16,16,0.372024,0.0,0.040831,0.000000e+00,0.064290,0.000000e+00,2.8125,0.0,0.0000,0.0,0.000,0.0,0.718650,0.0,0.767990,0.0,0.742070,0.0,0.262943,0.017289,0.312500,0.108253,0.593750,0.231419,0.972917,0.046910,0.315796,0.000000e+00,2.8125,0.0,1.666667,0.0,0.2500,0.0
3,hypergraphrag,3,16,16,16,0.297760,0.0,0.538288,0.000000e+00,0.363903,6.798700e-17,2.8125,0.0,0.1250,0.0,0.125,0.0,0.842689,0.0,0.867849,0.0,0.854512,0.0,0.320833,0.007217,0.333333,0.036084,0.641667,0.175928,0.928542,0.057201,0.324410,0.000000e+00,2.8125,0.0,2.666667,0.0,0.2500,0.0
4,llmbased,3,16,16,16,0.566587,0.0,0.188141,0.000000e+00,0.267850,0.000000e+00,2.8125,0.0,0.0000,0.0,0.000,0.0,0.825663,0.0,0.792304,0.0,0.807957,0.0,0.037500,0.010825,0.072917,0.062604,0.156250,0.115752,0.804167,0.078146,0.204821,3.399350e-17,2.8125,0.0,1.000000,0.0,0.1250,0.0



answer_winrate: 7 rows x 12 columns


,method,judge_count,comparisons_min,comparisons_max,wins_mean,wins_std,losses_mean,losses_std,ties_mean,ties_std,winrate_mean,winrate_std
0,LWE,2,618,618,475.5,26.162951,135.5,19.091883,7.0,7.071068,0.775081,0.036614
1,fullcontext,2,103,103,48.5,3.535534,53.5,4.949747,1.0,1.414214,0.475728,0.041191
2,grasp,2,103,103,15.5,4.949747,87.0,5.656854,0.5,0.707107,0.152913,0.051488
3,hipporag,2,103,103,11.5,9.192388,90.5,10.606602,1.0,1.414214,0.116505,0.096112
4,hypergraphrag,2,103,103,31.5,10.606602,68.5,6.363961,3.0,4.242641,0.320388,0.082381



pairwise_answer_winrate: 1236 rows x 19 columns


,ground_truth_id,ground_truth_qtype,question,ground_truth_answer,method_a,method_b,method_a_key,method_b_key,answer_a,answer_b,winner,judge_rationale,winning_method,method_a_score,method_b_score,judge_id,judge,evaluation_judge_id,evaluation_config_path
0,gt_0,"[""multi"", ""numeric""]","\nHow many ""experiment execution"" are there in...",The answer to the question is 15 unique execut...,fullcontext,LWE,fullcontext,ours,There are 13 experiment executions in the prov...,There are **15** experiment executions represe...,method_b,The answer provided by method_b is more unders...,LWE,0.0,1.0,old-llama-70b,Llama 3.3 70B,NaN,NaN
1,gt_1,"[""multi"", ""entity""]",\nIn what places do we utilize AI in this work...,\nThe ChatBS System utilizes AI for the follow...,fullcontext,LWE,fullcontext,ours,"```json { ""answer"": ""The workflow employs AI (...",AI is employed in several distinct stages of t...,method_b,The answer provided by method_b is more unders...,LWE,0.0,1.0,old-llama-70b,Llama 3.3 70B,NaN,NaN
2,gt_10,"[""multi"", ""entity""]",\nwhat are the input parameters of the functio...,\n The input parameters to sparql query fun...,fullcontext,LWE,fullcontext,ours,The SPARQL query builder function receives the...,The question asks for the *input parameters* o...,method_a,The answer from method_a is more understandabl...,fullcontext,1.0,0.0,old-llama-70b,Llama 3.3 70B,NaN,NaN
3,gt_100,"[""multi"", ""numeric""]","\nwhat is the number of output for the ""Output...","\n The number of output for the ""Output por...",fullcontext,LWE,fullcontext,ours,The execution produces a single output on the ...,The execution identified by **http://testwebsi...,method_b,Method B provides a more detailed and understa...,LWE,0.0,1.0,old-llama-70b,Llama 3.3 70B,NaN,NaN
4,gt_101,"[""multi"", ""numeric""]","\nwhat is the number of output for the ""Output...","\n The number of output for the ""Output por...",fullcontext,LWE,fullcontext,ours,The execution **id_20260420105727_300** produc...,The execution “http://testwebsite/testProgram#...,method_b,The answer from method_b is more faithful to t...,LWE,0.0,1.0,old-llama-70b,Llama 3.3 70B,NaN,NaN


## Inspect Available Columns

Use this cell when choosing the placeholders below.

In [4]:
for name, df in summaries.items():
    print(f"\n{name}")
    for column in df.columns:
        print(f"  - {column}")


bool
  - run
  - judge_count
  - evaluated_examples
  - evaluated_examples_min
  - evaluated_examples_max
  - answer_token_precision_mean
  - answer_token_precision_std
  - answer_token_recall_mean
  - answer_token_recall_std
  - answer_token_f1_mean
  - answer_token_f1_std
  - gt_entity_total_mean
  - gt_entity_total_std
  - gt_entity_covered_mean
  - gt_entity_covered_std
  - gt_entity_coverage_mean
  - gt_entity_coverage_std
  - bertscore_precision_mean
  - bertscore_precision_std
  - bertscore_recall_mean
  - bertscore_recall_std
  - bertscore_f1_mean
  - bertscore_f1_std
  - llm_completeness_mean
  - llm_completeness_std
  - llm_faithfulness_mean
  - llm_faithfulness_std
  - llm_relevance_mean
  - llm_relevance_std
  - llm_understanderbility_mean
  - llm_understanderbility_std
  - nli_entailment_max_mean
  - nli_entailment_max_std
  - bool_accuracy_mean
  - bool_accuracy_std

entity
  - run
  - judge_count
  - evaluated_examples
  - evaluated_examples_min
  - evaluated_examples_m

## Editable Paper Table Columns

Edit these lists to decide what goes into each paper table. Keep `run` or `method` first if you want method names visible.

In [5]:
# These columns show ensemble means and between-judge sample standard deviations.
bool_columns = [
    "run",
    "judge_count",
    "evaluated_examples",
    "bertscore_f1_mean",
    "bertscore_f1_std",
    "nli_entailment_max_mean",
    "nli_entailment_max_std",
    "bool_accuracy_mean",
    "bool_accuracy_std",
]

entity_columns = [
    "run",
    "judge_count",
    "evaluated_examples",
    "answer_token_f1_mean",
    "answer_token_f1_std",
    "bertscore_f1_mean",
    "bertscore_f1_std",
    "nli_entailment_max_mean",
    "nli_entailment_max_std",
    "entity_recall_final_mean",
    "entity_recall_final_std",
    "entity_precision_final_mean",
    "entity_precision_final_std",
    "entity_f1_final_mean",
    "entity_f1_final_std",
    "entity_recall_total_mean",
    "entity_recall_total_std",
    "entity_precision_total_mean",
    "entity_precision_total_std",
    "entity_f1_total_mean",
    "entity_f1_total_std",
]

numeric_columns = [
    "run",
    "judge_count",
    "evaluated_examples",
    "answer_token_f1_mean",
    "answer_token_f1_std",
    "numeric_accuracy_mean",
    "numeric_accuracy_std",
    "bertscore_f1_mean",
    "bertscore_f1_std",
    "nli_entailment_max_mean",
    "nli_entailment_max_std",
]

answer_winrate_columns = [
    "method",
    "judge_count",
    "comparisons_min",
    "comparisons_max",
    "winrate_mean",
    "winrate_std",
]

In [6]:
# Optional display names. Add, remove, or rename as needed for the paper.
column_labels = {
    "run": "Method",
    "method": "Method",
    "judge_count": "Judges",
    "evaluated_examples": "N",
    "comparisons_min": "Comparisons Min",
    "comparisons_max": "Comparisons Max",
    "answer_token_f1_mean": "Answer Token F1",
    "answer_token_f1_std": "Answer Token F1 SD",
    "bertscore_f1_mean": "BERTScore F1",
    "bertscore_f1_std": "BERTScore F1 SD",
    "nli_entailment_max_mean": "NLI Entailment",
    "nli_entailment_max_std": "NLI Entailment SD",
    "bool_accuracy_mean": "Bool Accuracy",
    "bool_accuracy_std": "Bool Accuracy SD",
    "entity_recall_final_mean": "Entity Recall",
    "entity_recall_final_std": "Entity Recall SD",
    "entity_precision_final_mean": "Entity Precision",
    "entity_precision_final_std": "Entity Precision SD",
    "entity_f1_final_mean": "Entity F1",
    "entity_f1_final_std": "Entity F1 SD",
    "entity_recall_total_mean": "Entity Recall Total",
    "entity_recall_total_std": "Entity Recall Total SD",
    "entity_precision_total_mean": "Entity Precision Total",
    "entity_precision_total_std": "Entity Precision Total SD",
    "entity_f1_total_mean": "Entity F1 Total",
    "entity_f1_total_std": "Entity F1 Total SD",
    "numeric_accuracy_mean": "Numeric Accuracy",
    "numeric_accuracy_std": "Numeric Accuracy SD",
    "winrate_mean": "Win Rate",
    "winrate_std": "Win Rate SD",
}

method_order = ["GWB", "VSB", "GRASP", "HippoRAG", "HyperGRAG", "Ours"]

method_labels = {
    "fullcontext": "FCB",
    "grasp": "GRASP",
    "hipporag": "HippoRAG",
    "hypergraphrag": "HyperGRAG",
    "llmbased": "GWB",
    "LWE": "Ours",
    "ours": "Ours",
    "vectorsimilarity": "VSB",
}

def order_method_rows(table, method_column="Method"):
    ordered = table[table[method_column].isin(method_order)].copy()
    ordered[method_column] = pd.Categorical(
        ordered[method_column],
        categories=method_order,
        ordered=True,
    )
    sort_columns = [method_column]
    if "Question Type" in ordered.columns:
        sort_columns = ["Question Type", method_column]
    return ordered.sort_values(sort_columns).reset_index(drop=True)

count_columns = {"Judges", "N", "Comparisons Min", "Comparisons Max"}

def scale_result_columns(table):
    scaled = table.copy()
    for position, column in enumerate(scaled.columns):
        if column in count_columns:
            continue
        values = scaled.iloc[:, position]
        if pd.api.types.is_numeric_dtype(values):
            scaled.iloc[:, position] = values * 100
    return scaled

## Build Individual Tables

In [7]:
round_digits = 3

bool_table = summaries["bool"][bool_columns].copy()
bool_table["run"] = bool_table["run"].replace(method_labels)
bool_table = scale_result_columns(bool_table.rename(columns=column_labels)).round(round_digits)
bool_table = order_method_rows(bool_table)

entity_table = summaries["entity"][entity_columns].copy()
entity_table["run"] = entity_table["run"].replace(method_labels)
entity_table = scale_result_columns(entity_table.rename(columns=column_labels)).round(round_digits)
entity_table = order_method_rows(entity_table)

numeric_table = summaries["numeric"][numeric_columns].copy()
numeric_table["run"] = numeric_table["run"].replace(method_labels)
numeric_table = scale_result_columns(numeric_table.rename(columns=column_labels)).round(round_digits)
numeric_table = order_method_rows(numeric_table)



In [8]:
if "answer_winrate" in summaries:
    answer_winrate_table = summaries["answer_winrate"][answer_winrate_columns].copy()
    answer_winrate_table["method"] = answer_winrate_table["method"].replace(method_labels)
    answer_winrate_table = scale_result_columns(answer_winrate_table.rename(columns=column_labels)).round(round_digits)
    answer_winrate_table = order_method_rows(answer_winrate_table)
else:
    answer_winrate_table = pd.DataFrame(
        columns=[column_labels.get(column, column) for column in answer_winrate_columns]
    )

In [9]:
display(bool_table)
display(entity_table)
display(numeric_table)
display(answer_winrate_table)

,Method,Judges,N,BERTScore F1,BERTScore F1 SD,NLI Entailment,NLI Entailment SD,Bool Accuracy,Bool Accuracy SD
0,GWB,3,38,77.851,0.0,27.776,0.0,0.000,0.0
1,VSB,3,38,79.052,0.0,47.551,0.0,34.211,0.0
2,GRASP,3,38,80.011,0.0,49.270,0.0,28.947,0.0
3,HippoRAG,3,38,73.511,0.0,26.060,0.0,52.632,0.0
4,HyperGRAG,3,38,81.902,0.0,50.359,0.0,42.105,0.0
5,Ours,3,38,83.548,0.0,48.683,0.0,71.053,0.0


,Method,Judges,N,Answer Token F1,Answer Token F1 SD,BERTScore F1,BERTScore F1 SD,NLI Entailment,NLI Entailment SD,Entity Recall,Entity Recall SD,Entity Precision,Entity Precision SD,Entity F1,Entity F1 SD,Entity Recall Total,Entity Recall Total SD,Entity Precision Total,Entity Precision Total SD,Entity F1 Total,Entity F1 Total SD
0,GWB,3,49,14.376,0.0,78.107,0.0,14.596,0.0,29.897,0.000,4.529,0.0,16.355,0.0,56.835,0.000,8.553,0.0,19.674,0.0
1,VSB,3,49,19.474,0.0,78.803,0.0,23.268,0.0,10.594,0.000,1.868,0.0,15.837,0.0,18.928,0.000,2.499,0.0,13.393,0.0
2,GRASP,3,49,9.741,0.0,80.548,0.0,26.018,0.0,0.904,1.566,17.500,NaN,58.333,NaN,0.904,1.566,17.500,NaN,58.333,NaN
3,HippoRAG,3,49,11.626,0.0,74.900,0.0,20.158,0.0,34.546,0.000,0.949,0.0,4.439,0.0,64.793,0.000,1.337,0.0,4.196,0.0
4,HyperGRAG,3,49,16.274,0.0,80.515,0.0,28.659,0.0,0.000,0.000,NaN,NaN,NaN,NaN,0.000,0.000,NaN,NaN,NaN,NaN
5,Ours,3,49,22.130,0.0,80.482,0.0,28.580,0.0,38.372,0.000,1.726,0.0,4.097,0.0,49.419,0.000,1.368,0.0,5.248,0.0


,Method,Judges,N,Answer Token F1,Answer Token F1 SD,Numeric Accuracy,Numeric Accuracy SD,BERTScore F1,BERTScore F1 SD,NLI Entailment,NLI Entailment SD
0,GWB,3,16,26.785,0.0,12.50,0.0,80.796,0.0,20.482,0.0
1,VSB,3,16,24.688,0.0,6.25,0.0,79.533,0.0,43.944,0.0
2,GRASP,3,16,5.524,0.0,12.50,0.0,80.487,0.0,46.511,0.0
3,HippoRAG,3,16,6.429,0.0,25.00,0.0,74.207,0.0,31.580,0.0
4,HyperGRAG,3,16,36.390,0.0,25.00,0.0,85.451,0.0,32.441,0.0
5,Ours,3,16,50.375,0.0,43.75,0.0,86.751,0.0,48.304,0.0


,Method,Judges,Comparisons Min,Comparisons Max,Win Rate,Win Rate SD
0,GWB,2,103,103,12.136,4.806
1,VSB,2,103,103,16.262,6.522
2,GRASP,2,103,103,15.291,5.149
3,HippoRAG,2,103,103,11.650,9.611
4,HyperGRAG,2,103,103,32.039,8.238
5,Ours,2,618,618,77.508,3.661


## Combined Table Placeholder

This creates one compact table across question types. Edit `combined_columns` to choose the shared metrics.

In [10]:
combined_columns = [
    "task",
    "run",
    "judge_count",
    "evaluated_examples",
    "answer_token_f1_mean",
    "answer_token_f1_std",
    "bertscore_f1_mean",
    "bertscore_f1_std",
    "nli_entailment_max_mean",
    "nli_entailment_max_std",
]

combined_parts = []
for task_name in ["bool", "entity", "numeric"]:
    part = summaries[task_name].copy()
    part.insert(0, "task", task_name)
    combined_parts.append(part)

combined_table = pd.concat(combined_parts, ignore_index=True)
combined_table = combined_table[combined_columns].copy()
combined_table["run"] = combined_table["run"].replace(method_labels)
combined_table = scale_result_columns(combined_table.rename(columns={**column_labels, "task": "Question Type"})).round(round_digits)
combined_table = order_method_rows(combined_table)

display(combined_table)

,Question Type,Method,Judges,N,Answer Token F1,Answer Token F1 SD,BERTScore F1,BERTScore F1 SD,NLI Entailment,NLI Entailment SD
0,bool,GWB,3,38,20.500,0.0,77.851,0.0,27.776,0.0
1,bool,VSB,3,38,28.414,0.0,79.052,0.0,47.551,0.0
2,bool,GRASP,3,38,8.512,0.0,80.011,0.0,49.270,0.0
3,bool,HippoRAG,3,38,0.000,0.0,73.511,0.0,26.060,0.0
4,bool,HyperGRAG,3,38,25.889,0.0,81.902,0.0,50.359,0.0
5,bool,Ours,3,38,32.231,0.0,83.548,0.0,48.683,0.0
6,entity,GWB,3,49,14.376,0.0,78.107,0.0,14.596,0.0
7,entity,VSB,3,49,19.474,0.0,78.803,0.0,23.268,0.0
8,entity,GRASP,3,49,9.741,0.0,80.548,0.0,26.018,0.0
9,entity,HippoRAG,3,49,11.626,0.0,74.900,0.0,20.158,0.0


## Overall Summary Table

This combines bool, entity, and numeric summaries into one method-level table using only columns common to all three categories. Metric means are weighted by the number of evaluated examples in each category.


In [11]:
category_names = CATEGORY_NAMES
common_summary_columns = set.intersection(
    *(set(judge_sources[category].columns) for category in category_names)
)

overall_summary_columns = [
    "run",
    "evaluated_examples",
    "answer_token_f1_mean",
    "bertscore_f1_mean",
    "bertscore_f1_std",
    "nli_entailment_max_mean",
    "nli_entailment_max_std",
    "llm_completeness_mean",
    "llm_completeness_std",
    "llm_faithfulness_mean",
    "llm_faithfulness_std",
    "llm_relevance_mean",
    "llm_relevance_std",
    "llm_understanderbility_mean",
    "llm_understanderbility_std",
]
overall_summary_columns = [
    column for column in overall_summary_columns
    if column in common_summary_columns
    and (column in {"run", "evaluated_examples"} or column.endswith("_mean"))
]

overall_summary_source = pd.concat(
    [
        judge_sources[category][["judge_id", *overall_summary_columns]].assign(task=category)
        for category in category_names
    ],
    ignore_index=True,
)

overall_metric_columns = [
    column for column in overall_summary_columns
    if column not in {"run", "evaluated_examples"}
]

overall_rows = []
for (judge_id, run_name), run_group in overall_summary_source.groupby(["judge_id", "run"], dropna=False):
    weights = pd.to_numeric(
        run_group["evaluated_examples"],
        errors="coerce",
    ).fillna(0)
    row = {
        "judge_id": judge_id,
        "run": run_name,
        "evaluated_examples": weights.sum(),
    }

    for column in overall_metric_columns:
        values = pd.to_numeric(run_group[column], errors="coerce")
        valid = values.notna() & weights.gt(0)
        if valid.any():
            row[column] = (values[valid] * weights[valid]).sum() / weights[valid].sum()
        else:
            row[column] = values.mean()

    overall_rows.append(row)

overall_judge_source = pd.DataFrame(overall_rows)
overall_summary_table = aggregate_judge_values(
    overall_judge_source,
    "run",
    overall_metric_columns,
)
overall_summary_table["run"] = overall_summary_table["run"].replace(method_labels)
overall_summary_table = scale_result_columns(overall_summary_table.rename(columns=column_labels)).round(round_digits)
overall_summary_table = order_method_rows(overall_summary_table)

display(overall_summary_table)


,Method,Judges,N,evaluated_examples_min,evaluated_examples_max,Answer Token F1,Answer Token F1 SD,BERTScore F1,BERTScore F1 SD,NLI Entailment,NLI Entailment SD,llm_completeness_mean,llm_completeness_std,llm_faithfulness_mean,llm_faithfulness_std,llm_relevance_mean,llm_relevance_std,llm_understanderbility_mean,llm_understanderbility_std
0,GWB,3,103,10300,10300,18.563,0.0,78.430,0.0,20.373,0.0,13.652,1.042,13.569,3.448,27.490,7.664,69.913,9.094
1,VSB,3,103,10300,10300,23.582,0.0,79.008,0.0,35.439,0.0,21.489,0.521,27.687,5.138,53.671,9.153,89.108,4.544
2,GRASP,3,103,10300,10300,8.633,0.0,80.340,0.0,37.780,0.0,29.223,18.450,32.715,15.979,46.171,21.276,67.513,24.367
3,HippoRAG,3,103,10300,10300,6.530,0.0,74.280,0.0,24.110,0.0,26.090,1.992,29.450,1.633,62.888,2.951,87.021,4.217
4,HyperGRAG,3,103,10300,10300,22.946,0.0,81.793,0.0,37.252,0.0,37.127,4.639,44.659,9.563,59.426,4.518,87.476,6.227
5,Ours,3,103,10300,10300,30.244,0.0,82.587,0.0,39.060,0.0,66.073,5.128,64.142,7.848,88.485,5.986,81.346,7.748


## Export Tables

Exports CSV and LaTeX versions into `paper_tables/`. Comment out any table you do not need.

In [12]:
tables = {
    "bool_summary": bool_table,
    "entity_summary": entity_table,
    "numeric_summary": numeric_table,
    "answer_winrate_summary": answer_winrate_table,
    "combined_summary": combined_table,
    "overall_summary": overall_summary_table,
}

for table_name, table in tables.items():
    csv_path = TABLE_OUTPUT_DIR / f"{table_name}.csv"
    tex_path = TABLE_OUTPUT_DIR / f"{table_name}.tex"

    table.to_csv(csv_path, index=False)
    table.to_latex(tex_path, index=False, escape=False)

    print(f"Wrote {csv_path}")
    print(f"Wrote {tex_path}")


Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/chatbs-base/paper_tables/judges/ensemble/bool_summary.csv
Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/chatbs-base/paper_tables/judges/ensemble/bool_summary.tex
Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/chatbs-base/paper_tables/judges/ensemble/entity_summary.csv
Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/chatbs-base/paper_tables/judges/ensemble/entity_summary.tex
Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/chatbs-base/paper_tables/judges/ensemble/numeric_summary.csv
Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/chatbs-base/paper_tables/judges/ensemble/numeric_summary.tex
Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/chatbs-base/paper_tables/judges/ensemble/answer_winrate_summary.csv
Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/chatbs-base/paper_tables/judges/ensemb

## Optional: Pairwise Win Rate Details

In [13]:
if "pairwise_answer_winrate" in summaries:
    pairwise_table = summaries["pairwise_answer_winrate"].copy()
    for column in pairwise_table.columns:
        if pairwise_table[column].dtype == "object":
            pairwise_table[column] = pairwise_table[column].replace(method_labels)
    pairwise_table = scale_result_columns(pairwise_table).round(round_digits)
    display(pairwise_table)

,ground_truth_id,ground_truth_qtype,question,ground_truth_answer,method_a,method_b,method_a_key,method_b_key,answer_a,answer_b,winner,judge_rationale,winning_method,method_a_score,method_b_score,judge_id,judge,evaluation_judge_id,evaluation_config_path
0,gt_0,"[""multi"", ""numeric""]","\nHow many ""experiment execution"" are there in...",The answer to the question is 15 unique execut...,fullcontext,LWE,fullcontext,ours,There are 13 experiment executions in the prov...,There are **15** experiment executions represe...,method_b,The answer provided by method_b is more unders...,LWE,0.0,100.0,old-llama-70b,Llama 3.3 70B,NaN,NaN
1,gt_1,"[""multi"", ""entity""]",\nIn what places do we utilize AI in this work...,\nThe ChatBS System utilizes AI for the follow...,fullcontext,LWE,fullcontext,ours,"```json { ""answer"": ""The workflow employs AI (...",AI is employed in several distinct stages of t...,method_b,The answer provided by method_b is more unders...,LWE,0.0,100.0,old-llama-70b,Llama 3.3 70B,NaN,NaN
2,gt_10,"[""multi"", ""entity""]",\nwhat are the input parameters of the functio...,\n The input parameters to sparql query fun...,fullcontext,LWE,fullcontext,ours,The SPARQL query builder function receives the...,The question asks for the *input parameters* o...,method_a,The answer from method_a is more understandabl...,fullcontext,100.0,0.0,old-llama-70b,Llama 3.3 70B,NaN,NaN
3,gt_100,"[""multi"", ""numeric""]","\nwhat is the number of output for the ""Output...","\n The number of output for the ""Output por...",fullcontext,LWE,fullcontext,ours,The execution produces a single output on the ...,The execution identified by **http://testwebsi...,method_b,Method B provides a more detailed and understa...,LWE,0.0,100.0,old-llama-70b,Llama 3.3 70B,NaN,NaN
4,gt_101,"[""multi"", ""numeric""]","\nwhat is the number of output for the ""Output...","\n The number of output for the ""Output por...",fullcontext,LWE,fullcontext,ours,The execution **id_20260420105727_300** produc...,The execution “http://testwebsite/testProgram#...,method_b,The answer from method_b is more faithful to t...,LWE,0.0,100.0,old-llama-70b,Llama 3.3 70B,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1231,gt_95,"[""multi"", ""numeric""]","\nwhat is the number of output for the ""Output...","\n The number of output for the ""Output por...",grasp,LWE,grasp,ours,NaN,The execution `http://testwebsite/testProgram#...,method_b,Method B matches the ground truth by clearly s...,LWE,0.0,100.0,gpt-5.4-mini,GPT-5.4 mini,gpt-5.4-mini,/home/desild/work/research/LLM-Workflow-Explor...
1232,gt_96,"[""multi"", ""numeric""]","\nwhat is the number of output for the ""Output...","\n The number of output for the ""Output por...",grasp,LWE,grasp,ours,<|channel|>final <|constrain|>answer<|message|...,The execution identified by `http://testwebsit...,method_b,Method B matches the ground truth by clearly s...,LWE,0.0,100.0,gpt-5.4-mini,GPT-5.4 mini,gpt-5.4-mini,/home/desild/work/research/LLM-Workflow-Explor...
1233,gt_97,"[""multi"", ""numeric""]","\nwhat is the number of output for the ""Output...","\n The number of output for the ""Output por...",grasp,LWE,grasp,ours,NaN,The execution identified by `http://testwebsit...,method_b,Method B correctly states that the output coun...,LWE,0.0,100.0,gpt-5.4-mini,GPT-5.4 mini,gpt-5.4-mini,/home/desild/work/research/LLM-Workflow-Explor...
1234,gt_98,"[""multi"", ""numeric""]","\nwhat is the number of output for the ""Output...","\n The number of output for the ""Output por...",grasp,LWE,grasp,ours,NaN,The execution `http://testwebsite/testProgram#...,method_a,Method B is incorrect because it says the outp...,grasp,100.0,0.0,gpt-5.4-mini,GPT-5.4 mini,gpt-5.4-mini,/home/desild/work/research/LLM-Workflow-Explor...
